# Ops 日频面板数据源：白名单挂载与因子计算测试

本 notebook 验证 `SmartQuantDataProvider(include_tables=[...])` 白名单能力：
只挂载 `ops_daily` 与 `ops_intra` 两张 cephfs 日频 parquet 面板
（资产轴与代码映射由 Provider 代码注册表提供，无需挂载行情表），
并用其中的特征计算一个简单因子。

覆盖的引擎能力：

- `include_tables` 白名单：分钟表、转债表、基本面表一律不挂载、不扫描；
- `parquet_panel` reader：每日一个 parquet 文件的日频面板（S=1）；
- `secu_code` 代码身份：物理文件按 SecuCode，经编译期冻结的
  `SmartQuant.InnerCode_SecuCode` 映射翻译回 InnerCode 内部协议；
- LoadNormalizer 统一数组协议与 FormulaBatch 多输出共享 DAG。


## 1. 导入公共 API


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_project_root(start=None):
    """向上查找包含新版 src/factor_engine 包的项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "factor_engine").exists():
            return path
    raise RuntimeError("找不到包含 src/factor_engine 的项目根目录")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from factor_engine import (  # noqa: E402
    BatchFactorEngine,
    ComputeRequest,
    DomainSpec,
    FormulaBatch,
    SmartQuantDataProvider,
    source,
)


## 2. 参数与白名单

白名单按 `source_tables` 条目的 `name` 过滤。资产轴与代码映射来自
Provider 代码注册表（`_ASSET_AXES`/`_CODE_MAPS`），不属于可挂载数据源，
因此只挂两张 ops 面板即可建任务，也不会与行情宽表发生字段撞名。


In [ ]:
START = "2026-08-17"
END = "2026-08-28"

INCLUDE_TABLES = ["ops_daily", "ops_intra"]
CHUNK_SIZE = 5
OUTPUT_CSV = Path("/tmp/ops_panel_factor_test.csv")


## 3. 创建只挂载白名单表的 DataProvider


In [ ]:
provider = SmartQuantDataProvider(include_tables=INCLUDE_TABLES)

print("provider:", type(provider).__name__)
print("catalog fingerprint:", provider.catalog_fingerprint)
print("catalog source count:", provider.catalog.source_count)

# 确认分钟/转债表未被挂载
mounted_keys = sorted(provider.catalog.sources)
assert not any(k.startswith(("stk.1min", "cb.")) for k in mounted_keys)
print("no minute/cb sources mounted: OK")

# 资产轴来自注册表而非挂载表：未挂任何轴表也能冻结任务资产轴

# 资产轴来自注册表而非挂载表，将在编译期随任务日期（含 lookback）冻结


## 4. 检查两个 ops 面板注册的特征

字段由 Catalog 扫描样例 parquet 自动注册为 `stk.1d.<列名>`；
这里打印全部已注册键并确认本公式要用的两个特征存在。


In [ ]:
REQUIRED_KEYS = [
    "stk.1d.ret_mod_1min",   # ops_daily
    "stk.1d.ret_Avg_[1]",    # ops_intra
]

print("mounted stk.1d keys:")
for key in mounted_keys:
    if key.startswith("stk.1d."):
        print(" ", key)

source_specs = provider.describe_many([source(key) for key in REQUIRED_KEYS])
source_df = pd.DataFrame(
    [
        {
            "key": ref.logical_key,
            "asset": spec.asset_type,
            "frequency": spec.frequency,
            "step_count": spec.step_count,
            "value_kind": spec.value_kind.value,
        }
        for ref, spec in source_specs.items()
    ]
)
source_df


## 5. 定义并计算一个简单因子

刻意保持简单：两个 ops 特征分别做 5 日均值后相减，验证两条
secu_code 面板链路可以同时进入同一个 FormulaBatch。


In [ ]:
batch = FormulaBatch.from_text(
    common_inputs="""
        daily_feat = source("stk.1d.ret_mod_1min")
        intra_feat = source("stk.1d.ret_Avg_[1]")
    """,
    formulas={
        "ops_blend": """
            factor = ts_mean(intra_feat, 5) - ts_mean(daily_feat, 5)
        """,
        "intra_raw": "factor = intra_feat",
    },
)

request = ComputeRequest(
    domain=DomainSpec(
        start=START,
        end=END,
        asset_scope={"stk": "all"},
        target_asset="stk",
        target_freq="1d",
        target_step_count=1,
    ),
    batch=batch,
)

result = BatchFactorEngine(provider).compute(request)
print("domain:", result.domain.shape, "dates", START, "->", END)
print("load calls:", result.stats.load_calls)


## 6. 查看结果

结果是共享输出域上的 `T × N × 1` 数组；转成 DataFrame 查看覆盖率与最新截面。


In [ ]:
result_dfs = {
    formula_id: pd.DataFrame(
        values[:, :, 0],
        index=pd.Index(result.domain.dates.astype(str), name="DataDate"),
        columns=pd.Index(result.domain.codes, name="InnerCode"),
    )
    for formula_id, values in result.arrays.items()
}

coverage = result_dfs["ops_blend"].notna().mean(axis=1)
print("ops_blend daily coverage:")
print(coverage)

latest = pd.DataFrame(
    {fid: frame.iloc[-1] for fid, frame in result_dfs.items()}
)
print("latest cross-section:", result.domain.dates[-1])
display(latest.dropna().head(10))
display(latest.describe())


## 7. 导出长表


In [ ]:
n_dates = len(result.domain.dates)
n_codes = len(result.domain.codes)
full_result_df = pd.DataFrame(
    {
        "runner_date": np.repeat(result.domain.dates.astype(str), n_codes),
        "InnerCode": np.tile(result.domain.codes, n_dates),
    }
)
for formula_id, frame in result_dfs.items():
    full_result_df[formula_id] = frame.to_numpy(copy=False).reshape(-1)

if OUTPUT_CSV is not None:
    full_result_df.to_csv(OUTPUT_CSV, index=False)
    print("saved:", OUTPUT_CSV)
full_result_df.head()
